## RAG System Lab

This notebook guides you through building a Retrieval-Augmented Generation (RAG) system using LangChain, ChromaDB, and a Language Model.

### Installation Commands

First, let's install all the necessary libraries.

In [ ]:
!pip uninstall -y langchain langchain-community langchain-openai langchain-huggingface chromadb sentence-transformers openai
!pip install langchain chromadb sentence-transformers openai langchain-community langchain-openai langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 16.2 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.30.0
    Uninstalling openai-2.30.0:
      Successfully uninstalled openai-2.30.0
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.8
    Uninstalling langgraph-prebuilt-1.0.8:
      Successfully uninstalled langgraph-prebuilt-1.0.8
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.1.4
    Uninstalling langgraph-1.1.4:
      Successfully uninstalled langgraph-1.1.4
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.14
    Uninstalling langchain-1.2.14:
      Successfully uninstalled langchain-1.2.14


### 🔹 STEP 1 — Import Required Libraries

Here, we import the necessary components from LangChain and other libraries to build our RAG system.

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_openai import OpenAI
from langchain.chains.retrieval import RetrievalQA

ModuleNotFoundError: No module named 'langchain.chains'

#### Student Explanation

*   **What is LangChain?**
    LangChain is a framework designed to simplify the creation of applications using large language models (LLMs). It provides tools, components, and interfaces to chain together different LLM functionalities, such as prompt management, memory, document loading, and integration with external data sources.

*   **Why is it used in RAG systems?**
    In RAG systems, LangChain acts as an orchestrator. It helps integrate various components like vector databases (for retrieval), LLMs (for generation), and custom logic into a cohesive pipeline. It simplifies the process of getting relevant information from a knowledge base and feeding it to an LLM to generate more informed and accurate answers, reducing hallucinations.

### 🔹 STEP 2 — Load Embedding Model

An embedding model converts text into numerical vectors, which are crucial for finding similar documents.

In [ ]:
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

/tmp/ipykernel_1522/2537902316.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

#### Explain

*   **Difference between embeddings and raw text**
    **Raw text** is human-readable language (words, sentences). **Embeddings** are numerical representations (vectors) of text. These vectors capture the semantic meaning of the text, meaning words or phrases with similar meanings will have similar vector representations in a multi-dimensional space.

*   **Role of embeddings in retrieval**
    Embeddings are fundamental for retrieval. When a query comes in, it's converted into an embedding. This query embedding is then compared against the embeddings of all documents in the knowledge base. Documents whose embeddings are 'closest' (most similar) to the query embedding are considered most relevant and are retrieved. This allows for semantic search, finding documents based on meaning rather than just keyword matching.

### 🔹 STEP 3 — Load Data into ChromaDB via LangChain

We will now load some sample documents into ChromaDB, a vector store, using LangChain's integration.

In [ ]:
documents = [
    "Artificial Intelligence is the simulation of human intelligence.",
    "Machine Learning is a subset of AI.",
    "Deep Learning uses neural networks.",
    "Natural Language Processing deals with text data.",
    "RAG combines retrieval and generation."
]

db = Chroma.from_texts(documents, embedding_model)

#### Explain

*   **How LangChain simplifies vector DB operations**
    LangChain provides a higher-level abstraction for interacting with various vector databases like ChromaDB. Instead of writing low-level code to embed documents, store them, and query them, LangChain's `Chroma.from_texts` (and similar methods for other DBs) handles the embedding process, indexing, and storage automatically. It streamlines the creation and management of vector stores, making it easier to integrate them into complex LLM applications.

*   **Difference between direct ChromaDB vs LangChain wrapper**
    **Direct ChromaDB** involves using the ChromaDB client library directly. This gives fine-grained control over every operation but requires more boilerplate code for embedding, adding, and querying. You would manually manage the embedding process before adding to Chroma.
    **LangChain wrapper** (e.g., `langchain_community.vectorstores.Chroma`) provides a simplified interface. It often integrates the embedding process directly, allowing you to pass raw text and an embedding model, and it handles the conversion and storage transparently. This reduces complexity and allows developers to focus on the application logic rather than database specifics.

### 🔹 STEP 4 — Create Retriever

The retriever is responsible for fetching relevant documents from the vector store based on a given query.

In [ ]:
retriever = db.as_retriever()

#### Explain

*   **What is a retriever?**
    A retriever is a component in a RAG system that identifies and fetches the most relevant pieces of information (documents, chunks of text) from a knowledge base or vector store in response to a user's query. Its primary goal is to provide context to the LLM.

*   **How it fetches relevant documents**
    When a query is received, the retriever typically converts the query into an embedding using the same embedding model used for the documents. It then performs a similarity search within the vector store to find documents whose embeddings are most similar to the query's embedding. These top-k (e.g., top 3 or 5) most similar documents are then returned as relevant context.

### 🔹 STEP 5 — Initialize LLM

We initialize the Large Language Model (LLM) that will generate answers. Here, we'll use OpenAI's LLM. Remember to set your OpenAI API key securely.

In [ ]:
import os
from google.colab import userdata

# Securely store your OpenAI API key in Colab Secrets and retrieve it.
# Go to the '🔑' icon on the left panel, click 'Add new secret',
# enter 'OPENAI_API_KEY' as the name and paste your API key as the value.
# Then, ensure 'Notebook access' is enabled for this secret.

# os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY') # Uncomment and run if you have an OpenAI API key

# If you don't have an OpenAI API key or want to test without it,
# you can use a mock LLM or comment out the llm initialization for now.
# For demonstration, we'll proceed assuming an API key might be set,
# but if not, the subsequent steps involving the LLM will fail.

# Initialize OpenAI LLM
# Note: If you don't provide an API key, this will raise an error.
# For testing without an API key, you might need to use a different LLM provider
# or a local model, or a mock LLM for development purposes.
llm = OpenAI(temperature=0) # temperature=0 makes the output more deterministic

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

### 🔹 STEP 6 — Build Retrieval QA Chain

This step combines the retriever and the LLM into a single chain, allowing the system to first retrieve relevant documents and then use them to generate an answer.

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever
)

NameError: name 'RetrievalQA' is not defined

#### Explain

*   **What is a chain in LangChain?**
    In LangChain, a "chain" is an end-to-end pipeline that combines different components (like LLMs, retrievers, parsers, prompt templates, etc.) to accomplish a specific task. Chains allow for modular and reusable logic, where the output of one component becomes the input of the next.

*   **How retrieval + generation are combined**
    The `RetrievalQA` chain exemplifies this combination:
    1.  **Retrieval Phase:** When a query is given to the `qa_chain`, it first passes the query to the `retriever`. The retriever searches the vector database and identifies the most relevant documents.
    2.  **Generation Phase:** These retrieved documents, along with the original query, are then passed to the `llm` as context. The LLM then uses this context to formulate an informed and relevant answer. This process ensures that the LLM's response is grounded in factual information from the knowledge base, rather than solely relying on its pre-trained knowledge.

### 🔹 STEP 7 — Query the System

Now, let's ask a question and see how our RAG system responds.

In [ ]:
query = "What is Artificial Intelligence?"
response = qa_chain.run(query)

print(response)

NameError: name 'qa_chain' is not defined

### 🔹 STEP 8 — Compare Retrieval vs RAG Output

It's important to understand what information was retrieved versus what the LLM generated as a final answer.

In [ ]:
docs = retriever.get_relevant_documents(query)

print("Retrieved Docs:")
for d in docs:
    print(d.page_content)

AttributeError: 'VectorStoreRetriever' object has no attribute 'get_relevant_documents'

#### Explain

*   **Difference between:**
    *   **Retrieved text:** This is the raw, unadulterated content directly pulled from your knowledge base (vector store) by the retriever. It consists of the most semantically similar documents to your query.
    *   **Generated answer:** This is the natural language response produced by the Large Language Model. The LLM takes the retrieved text as context, synthesizes the information, and then formulates a coherent and concise answer to the user's original query. The generated answer aims to be more conversational and directly address the question, potentially summarizing or rephrasing the retrieved information.

### 🔹 STEP 9 — Custom Function for QA

Encapsulate the querying logic into a reusable function for convenience.

In [ ]:
def ask_question(query):
    return qa_chain.run(query)

print(ask_question("Explain deep learning"))
print(ask_question("What is machine learning?"))

NameError: name 'qa_chain' is not defined

### 🔹 STEP 10 — RAG Workflow Explanation

Here's a step-by-step explanation of the RAG workflow:

1.  **User enters query:** The process begins when a user asks a question (e.g., "What is AI?").
2.  **Query converted into embedding:** The user's query is transformed into a numerical vector (embedding) using a pre-trained embedding model. This allows for semantic comparison with stored documents.
3.  **Retriever finds similar documents:** The query embedding is used to perform a similarity search in a vector database (like ChromaDB). The retriever identifies and fetches a set of documents or text chunks from the knowledge base that are most semantically relevant to the query.
4.  **Documents passed to LLM:** The retrieved relevant documents, along with the original user query, are then sent to a Large Language Model (LLM) as part of its input prompt. These documents provide factual context.
5.  **LLM generates final answer:** The LLM processes the query and the provided contextual documents. It uses this information to synthesize a comprehensive, accurate, and context-aware answer, which is then presented to the user. This approach helps reduce hallucinations and ensures the answer is grounded in the provided data.

### Additional Tasks (For Better Understanding)

#### 🔸 Task A: Change dataset topic

Try changing the `documents` list in **Step 3** to a different topic, for example, Healthcare or Finance, and observe how the responses change.

```python
# Example for Task A
# documents_healthcare = [
#     "Telemedicine allows remote patient consultations.",
#     "AI is transforming drug discovery.",
#     "Electronic health records improve patient data management."
# ]
# db_healthcare = Chroma.from_texts(documents_healthcare, embedding_model)
# retriever_healthcare = db_healthcare.as_retriever()
# qa_chain_healthcare = RetrievalQA.from_chain_type(llm=llm, retriever=retriever_healthcare)
# print(qa_chain_healthcare.run("What is telemedicine?"))
```

#### 🔸 Task B: Increase k value in retriever

The `k` parameter controls the number of most relevant documents the retriever fetches. Increasing `k` might provide more context to the LLM, but could also introduce irrelevant information. Let's try it:

In [ ]:
# Create a retriever that fetches 3 documents
retriever_k3 = db.as_retriever(search_kwargs={"k": 3})

qa_chain_k3 = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever_k3
)

print(f"Response with k=1 (default): {qa_chain.run('What is AI?')}")
print(f"Response with k=3: {qa_chain_k3.run('What is AI?')}")

# Also check the retrieved docs
print("\nRetrieved Docs with k=3 for 'What is AI?':")
for d in retriever_k3.get_relevant_documents('What is AI?'):
    print(d.page_content)

NameError: name 'RetrievalQA' is not defined

#### 🔸 Task C: Try different queries and observe output quality

Experiment with various questions to see how the system performs with different inputs and if it can handle questions not directly addressed in the training data but inferable from the context.

```python
# Example for Task C
# print(ask_question("How do neural networks relate to deep learning?"))
# print(ask_question("What is NLP?"))
# print(ask_question("Tell me about RAG."))
```

### Questions Students Must Answer

1.  **What is RAG architecture?**
    *Your Answer Here*

2.  **How does LangChain help in building RAG?**
    *Your Answer Here*

3.  **What is the role of retriever in RAG?**
    *Your Answer Here*

4.  **Difference between Retrieval system and Generative system?**
    *Your Answer Here*

5.  **Why combine retrieval with LLM?**
    *Your Answer Here*

6.  **What are limitations of RAG?**
    *Your Answer Here*

7.  **What is hallucination in LLMs?**
    *Your Answer Here*
